In [ ]:
#Prepare variables at daily scale for bias correction instead of downscaling
# we use chirps as target resolution

import os
import gc
import glob
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import pyarrow as pa
import pyarrow.parquet as pq
from scipy.spatial import cKDTree
from tqdm import tqdm

start_year, start_month, start_day = 2001, 1, 1
end_year,   end_month, end_day   = 2005, 12, 1

static_paths = {
    "cluster":         r"..\Data\Processed\Spatial_clustering\Precipitation_clusters.tif",
    "elev_mean":       r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_mean.tif",
    "elev_std":        r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_std.tif",
    "elev_max":        r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif",
    "elev_min":        r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_min.tif",
    "slope_mean":      r"..\Data\Datasets\DEM_1km\DEM_1km_slope_mean.tif",
    "aspect_mean":     r"..\Data\Datasets\DEM_1km\DEM_1km_aspect_mean.tif",
    "elev_smoothed":   r"..\Data\Processed\Fetch\Preprocessing\dem_smoothed.tif",
    "fetch_1000m_dist":r"..\Data\Processed\Fetch\Preprocessing\fetch_dist_1000m.tif",
    "fetch_1000m_dir": r"..\Data\Processed\Fetch\Preprocessing\fetch_dir_1000m.tif",
    "fetch_500m_dist": r"..\Data\Processed\Fetch\Preprocessing\fetch_dist_500m.tif",
    "fetch_500m_dir":  r"..\Data\Processed\Fetch\Preprocessing\fetch_dir_500m.tif",
    "local_relief":    r"..\Data\Processed\Fetch\Preprocessing\local_relief.tif",
}

# In-situ
insitu_csv      = r"..\Data\Processed\Insitu\insitu_rainfall_1991_2020.csv"
train_stations  = r"..\Data\Processed\Split_stations\train_stations.csv"
test_stations   = r"..\Data\Processed\Split_stations\test_stations.csv"

out_base   = r"..\\Data\\Processed\\Bias_correction_daily\\Prepared_dataframe"
daily_train_dir = os.path.join(out_base, "daily_dynamic_train_df")
daily_test_dir = os.path.join(out_base, "daily_dynamic_test_df")
parquet_train_dir  = os.path.join(out_base, "yearly_parquet_train")
parquet_test_dir   = os.path.join(out_base, "yearly_parquet_test")
static_dir         = os.path.join(out_base, "static_df")

for d in [out_base, daily_train_dir, daily_test_dir, parquet_train_dir,
          parquet_test_dir, static_dir]:
    os.makedirs(d, exist_ok=True)

template_path = r"..\Data\Datasets\CHIRPS_prec\CHIRPS_prec_19910101.tif"
template = rxr.open_rasterio(template_path)

In [ ]:
# HELPERS

def doy_cyclical(year: int, doy: int):
    """Return (sin, cos) encoding for the day of the year."""
    days_in_year = 366 if pd.Timestamp(year, 1, 1).is_leap_year else 365
    angle = 2 * np.pi * doy / days_in_year
    return np.sin(angle), np.cos(angle)

def day_label(year: int, month: int, day: int) -> str:
    return f"{year}{month:02d}{day:02d}"

def build_daily_paths(year: int, month: int, day: int) -> dict:
    """
    Return dict of {feature_name: filepath} for all daily rasters
    for the given year-month-day. Missing files are noted but not fatal —
    the loader handles absent paths with NaN fill.
    """
    return {
        "era5l": f"..\Data\Datasets\ERA5L_prec\ERA5L_prec_{year}{month:02d}{day:02d}.tif",
        "chirps": f"..\Data\Datasets\CHIRPS_prec\CHIRPS_prec_{year}{month:02d}{day:02d}.tif",
        "har": f"..\Data\Datasets\HAR_prec\HAR_prec_{year}{month:02d}{day:02d}.tif",
        "persiann_prec":  f"..\Data\Datasets\PERSIANN_prec\PERSIANN_precTot_{year}{month:02d}{day:02d}.tif",
        "mswep_prec":     f"..\Data\Datasets\MSWEP_prec\MSWEP_Prec_{year}{month:02d}{day:02d}.tif",
        "gldas_air_temp": f"..\\Data\\Datasets\\GLDAS\\air_temp\\GLDAS_air_temp_{year}{month:02d}{day:02d}.tif",
        "gldas_lwdown": f"..\Data\Datasets\GLDAS\lwdown\GLDAS_lw_down_{year}{month:02d}{day:02d}.tif",
        "gldas_specific_humidity": f"..\Data\Datasets\GLDAS\specific_humidity\GLDAS_specific_humidity_{year}{month:02d}{day:02d}.tif",
        "gldas_surface_pressure": f"..\Data\Datasets\GLDAS\surface_pressure\GLDAS_surface_pressure_{year}{month:02d}{day:02d}.tif",
        "gldas_swdown": f"..\Data\Datasets\GLDAS\swdown\GLDAS_sw_down_{year}{month:02d}{day:02d}.tif",
        "gldas_wind_speed": f"..\Data\Datasets\GLDAS\wind_speed\GLDAS_wind_speed_{year}{month:02d}{day:02d}.tif",
        "insitu_kriged": f"..\Data\Processed\Bias_correction_daily\Insitu_kriged_daily\insitu_precip_{year}{month:02d}{day:02d}.tif"
    }    

In [3]:
# 1. Static feature dataframe -------------------------------------

static_csv = os.path.join(static_dir, "static_df.csv")
 
if os.path.exists(static_csv):
    print("Loading cached static df...")
    static_df = pd.read_csv(static_csv)
else:
    print("Building static df from rasters...")
    aligned = []
    for name, path in static_paths.items():
        da = rxr.open_rasterio(path, masked=True).rio.write_crs("EPSG:4326")
        da = da.rio.reproject_match(template).squeeze("band", drop=True)
        da.name = name
        aligned.append(da)
    static_xr = xr.merge(aligned, compat="override")
    static_df  = static_xr.to_dataframe().reset_index()
    static_df.to_csv(static_csv, index=False)
    del aligned, static_xr
    gc.collect()
 
static_df[["x", "y"]] = static_df[["x", "y"]].round(5)
print(f"Static df: {len(static_df):,} pixels")

Building static df from rasters...
Static df: 15,753 pixels


In [4]:
# 2 Read insitu and separate train-test ---------------------------------

insitu_raw = pd.read_csv(insitu_csv)
insitu_raw["date"]  = pd.to_datetime(insitu_raw[["year", "month", "day"]])
insitu_raw  = insitu_raw.rename(columns={"lat": "y", "long": "x",
    "rainfall_sum": "insitu_prec"})
insitu_raw  = insitu_raw[insitu_raw["insitu_prec"] >= 0] 

# Filter for study period
insitu_daily = insitu_raw[
    (insitu_raw["year"]  >= start_year) &
    (insitu_raw["year"]  <= end_year)
].copy()
insitu_daily[["x", "y"]] = insitu_daily[["x", "y"]].round(5)

# Train / test split
train_ids = set(pd.read_csv(train_stations)["index_no"])
test_ids  = set(pd.read_csv(test_stations)["index_no"])

insitu_train_daily = insitu_daily[insitu_daily["index_no"].isin(train_ids)].copy()
insitu_test_daily =  insitu_daily[insitu_daily["index_no"].isin(test_ids)].copy()

In [ ]:
# 3. Main Loop daily ---------------------------------------------------------

daily_range = pd.period_range(
    start = f"{start_year}-{start_month:02d}-{start_day:02d}",
    end = f"{end_year}-{end_month:02d}-{end_day:02d}",
    freq= "D"
)

print(f"Processing {len(daily_range)} days")

for period in tqdm(daily_range, desc = "Daily feature extraction"):
    year = period.year
    month = period.month
    day = period.day
    label = day_label(year, month, day)
    
    train_out = os.path.join(daily_train_dir, f"merged_{label}.csv")
    test_out = os.path.join(daily_test_dir, f"merged_{label}.csv")

    if os.path.exists(train_out) and os.path.exists(test_out):
        continue 

    #get daily rasters and flatten to pixel df
    paths = build_daily_paths(year, month, day)
    aligned = []
    missing_vars = []

    for name, path in paths.items():
        if not os.path.exists(path):
            missing_vars.append(name)
            continue
        
        with rxr.open_rasterio(path, masked=True) as src:
            da = src.rio.write_crs("EPSG:4326")
            da = da.rio.reproject_match(template).squeeze("band", drop=True)
            da.name = name
            aligned.append(da.load())
      
    if missing_vars:
        tqdm.write(f"  [{label}] missing rasters: {missing_vars}")

    if not aligned:
        tqdm.write(f"  [{label}] no rasters found — skipping")
        continue

    # merge all dynamic rasters
    dynamic_xr = xr.merge(aligned, compat='override')
    dynamic_df = dynamic_xr.to_dataframe().reset_index()
    dynamic_df[["x", "y"]] = dynamic_df[["x", "y"]].round(5)

    for name in missing_vars:
        dynamic_df[name] = np.nan
 
    del aligned, dynamic_xr
    gc.collect()

    #merge with static features
    daily_grid = dynamic_df.merge(static_df, on = ["x", "y"], how = "inner")
    doy = pd.to_datetime({'year': [year], 'month': [month], 'day': [day]})[0].dayofyear
    sin_doy, cos_doy = doy_cyclical(year, doy)

    daily_grid['doy_sin'] = sin_doy
    daily_grid['doy_cos'] = cos_doy
    daily_grid['year'] = year
    daily_grid['month'] = month
    daily_grid['day'] = day

    del dynamic_df
    gc.collect()

    # cKDTree match to station locations 
    # Threshold 0.035°  half of the diagonal of one pixel tolerance
    insitu_train_d = insitu_train_daily[
        (insitu_train_daily["year"]  == year) &
        (insitu_train_daily["month"] == month) &
        (insitu_train_daily["day"] == day)
    ].copy()
 
    insitu_test_d = insitu_test_daily[
        (insitu_test_daily["year"]  == year) &
        (insitu_test_daily["month"] == month) &
        (insitu_test_daily["day"] == day)
    ].copy()
 
    if insitu_train_d.empty and insitu_test_d.empty:
        del daily_grid
        gc.collect()
        continue

    tree = cKDTree(daily_grid[['x','y']].values)

    # matching training stations
    if not insitu_train_d.empty:
        dist_tr, idx_tr = tree.query(insitu_train_d[["x", "y"]].values, k=1)
        mask_tr = dist_tr <= 0.035

        if mask_tr.any():
            station_rows = insitu_train_d.iloc[mask_tr].reset_index(drop = True)
            grid_rows = daily_grid.iloc[idx_tr[mask_tr]].reset_index(drop=True)

            final_train = pd.concat([
                station_rows[["index_no", "y", "x", "year", "month", "day", "insitu_prec"]],
                grid_rows.drop(columns=["x","y","year", "month", 'day'])
            ], axis = 1)

            final_train.to_csv(train_out, index=False)

    # matching training stations
    if not insitu_test_d.empty:
        dist_te, idx_te = tree.query(insitu_test_d[["x", "y"]].values, k=1)
        mask_te = dist_te <= 0.035

        if mask_te.any():
            station_rows = insitu_test_d.iloc[mask_te].reset_index(drop = True)
            grid_rows = daily_grid.iloc[idx_te[mask_te]].reset_index(drop=True)

            final_test = pd.concat([
                station_rows[["index_no", "y", "x", "year", "month", "day", "insitu_prec"]],
                grid_rows.drop(columns=["x","y", "year", "month", "day"])
            ], axis = 1)

            final_test.to_csv(test_out, index=False)
    
    del daily_grid, insitu_train_d, insitu_test_d
    gc.collect()

Processing 1796 days


Daily feature extraction:  98%|█████████▊| 1764/1796 [14:43<00:13,  2.38it/s]

  [20051030] missing rasters: ['era5l']


Daily feature extraction: 100%|██████████| 1796/1796 [14:58<00:00,  2.00it/s]


In [6]:
# 4. Aggregate daily csvs to yearly parquet to final combined df

def aggregate_to_parquet(daily_dir: str, parquet_dir:str, split_label:str): 
    """
    Read daily csvs year by year, write to parquet, then concatenate
    all years into a single final df. Returns path to final parquet
    """
    parquet_paths = []

    for year in range(start_year, end_year + 1):
        yearly_path = os.path.join(parquet_dir, f"full_{year}.parquet")
        daily_periods = pd.period_range(
            start = f"{year}-01-01", end=f"{year}-12-31", freq="D"
        )
        expected = [
            os.path.join(daily_dir, f"merged_{day_label(p.year, p.month, p.day)}.csv")
            for p in daily_periods
        ]
        found = [f for f in expected if os.path.exists(f)]

        if not found:
            print(f"    [{split_label}] {year} -- no daily files found, skipping")

        if found:
            yearly_df = pd.concat([pd.read_csv(fpath) for fpath in found], ignore_index=True)
            
            yearly_df.to_parquet(yearly_path, index=False)
            
            parquet_paths.append(yearly_path)
            print(f"    [{split_label}] {year} -- {len(found)} days written to parquet")
        else:
            print(f"    [{split_label}] {year} -- no daily files found, skipping")
            continue
    
    if not parquet_paths:
        print(f"    [{split_label}] {year} -- {len(found)} months written to parquet")
        return None

    final_df = pd.concat([pd.read_parquet(p) for p in parquet_paths], ignore_index = True)
    final_path = os.path.join(out_base, f"final_{split_label}_df.parquet")
    final_df.to_parquet(final_path, index = False)

    final_df.head(10_000).to_csv(
        os.path.join(out_base, f"final_{split_label}_df_preview.csv"), index=False
    )
 
    print(f"\n  [{split_label}] Final df: {len(final_df):,} rows × {len(final_df.columns)} cols")
    print(f"  Saved: {final_path}")
    return final_path

train_final_path = aggregate_to_parquet(daily_train_dir, parquet_train_dir, "train")
test_final_path = aggregate_to_parquet(daily_test_dir, parquet_test_dir, "test")

    [train] 2001 -- 365 days written to parquet
    [train] 2002 -- 365 days written to parquet
    [train] 2003 -- 365 days written to parquet
    [train] 2004 -- 366 days written to parquet
    [train] 2005 -- 335 days written to parquet

  [train] Final df: 86,683 rows × 37 cols
  Saved: ..\\Data\\Processed\\Bias_correction_daily\\Classification_model\final_train_df.parquet
    [test] 2001 -- 365 days written to parquet
    [test] 2002 -- 365 days written to parquet
    [test] 2003 -- 365 days written to parquet
    [test] 2004 -- 366 days written to parquet
    [test] 2005 -- 335 days written to parquet

  [test] Final df: 31,059 rows × 36 cols
  Saved: ..\\Data\\Processed\\Bias_correction_daily\\Classification_model\final_test_df.parquet


In [7]:
# ── 5. Sanity check on final training df ──────────────────────────────────────
 
if train_final_path:
    df = pd.read_parquet(train_final_path)
 
    print("\n" + "=" * 60)
    print("Training df sanity check")
    print("=" * 60)
    print(f"Shape              : {df.shape}")
    print(f"Columns            : {list(df.columns)}")
    print(f"\nStation-day combinations     : {len(df):,}")
    print(f"Unique stations    : {df['index_no'].nunique()}")
    print(f"Year range         : {df['year'].min()}–{df['year'].max()}")
    print(f"Days distribution :\n{df['year'].value_counts().sort_index()}")    


Training df sanity check
Shape              : (86683, 37)
Columns            : ['index_no', 'y', 'x', 'year', 'month', 'day', 'insitu_prec', 'spatial_ref_x', 'era5l', 'chirps', 'har', 'persiann_prec', 'mswep_prec', 'gldas_air_temp', 'gldas_lwdown', 'gldas_specific_humidity', 'gldas_surface_pressure', 'gldas_swdown', 'gldas_wind_speed', 'insitu_kriged', 'spatial_ref_y', 'cluster', 'elev_mean', 'elev_std', 'elev_max', 'elev_min', 'slope_mean', 'aspect_mean', 'elev_smoothed', 'fetch_1000m_dist', 'fetch_1000m_dir', 'fetch_500m_dist', 'fetch_500m_dir', 'local_relief', 'doy_sin', 'doy_cos', 'day.1']

Station-day combinations     : 86,683
Unique stations    : 50
Year range         : 2001–2005
Days distribution :
year
2001    17716
2002    18000
2003    17452
2004    17874
2005    15641
Name: count, dtype: int64
